# 🤗 Introduction to Hugging Face Transformers



### What We'll Cover in Today's Session :
* **Transformers:** Understand the core idea behind Transformer models
*  **Hugging Face Pipelines:** Perform sentiment analysis, text generation, summarization, translation, and more
* **Under the Hood:** Explore tokenizers and models
* **Hugging Face Hub:** Find, load, and swap between different models
*  **Fine-Tuning:** Train a model on a real-world dataset
*  **Gradio:** Turn your model into a shareable web demo


### Two Tracks
Sections marked **⭐ Core** are the topics we’ll cover together during today’s workshop (~90 minutes).

Sections marked **🎁 Bonus** are optional topics you can explore later at your own pace, or if we have extra time

### Before we start
1. Make sure you're signed into a Google account (for Colab)
2. **File → Save a copy in Drive** so your edits are saved
3. **Turn on a free GPU:** Runtime → Change runtime type → **T4 GPU**. Most sections work fine on CPU, but the fine-tuning section (Section 9) needs a GPU to finish in a few minutes.

##⭐ 0. How RNNs and Transformers Process Text

Both RNNs and Transformers process sequences of words, but they handle the sequence differently.

#### **RNN: One step at a time**

An RNN reads the sentence **sequentially**, carrying information from one step to the next.

```text
"I" → "love" → "Transformers" → "because" → "they're" → "powerful"
 ↓       ↓           ↓              ↓            ↓            ↓
[State] → [State] → [State] → [State] → [State] → [State]
```

Each new word is processed using the information carried from the previous words.

#### **Transformer: Look at the whole sequence**

A Transformer can process the tokens **in parallel** and use self-attention to determine which words are relevant to each other.

```text
"I"       ───────────────┐
"love"    ───────────────┤
"Transformers" ──────────┤ → Self-Attention → Contextual representations
"because" ───────────────┤
"they're" ───────────────┤
"powerful" ──────────────┘
```

#### **Key idea**

> **RNNs process sequentially. Transformers use self-attention to connect information across the sequence and can process tokens in parallel during training.**

Self-attention allows a word to directly consider other relevant words, even when they are far away in the sentence.


##⭐ 1. Explore Hugging Face

Now that we understand the basic idea behind Transformers, let's look at where we can actually find and use them.

**Hugging Face** is an open-source AI platform and community where you can find:

| Section         | What you'll find                                       |
| --------------- | ------------------------------------------------------ |
| **Models**   | Pretrained and fine-tuned machine-learning models      |
| **Datasets** | Datasets for training, evaluation, and experimentation |
|  **Spaces**   | Interactive demos and ML applications                  |

### **Useful links**

*  **Models:** [huggingface.co/models](https://huggingface.co/models)
*  **Datasets:** [huggingface.co/datasets](https://huggingface.co/datasets)
*  **Spaces:** [huggingface.co/spaces](https://huggingface.co/spaces)
*  **Tasks:** [huggingface.co/tasks](https://huggingface.co/tasks)
*  **Transformers Documentation:** [huggingface.co/docs/transformers](https://huggingface.co/docs/transformers)



## ⭐ 2. Setup

Let's install the `transformers` library (Colab has most dependencies already, this just makes sure we have the latest).


In [ ]:
!pip install -q transformers datasets accelerate gradio torch
print("Setup Complete")

In [ ]:
import torch
print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

> 💡 It's  fine if `GPU available: False` — everything in this workshop runs on CPU too, just a little slower.


## ⭐ 3. Your First Pipeline: Sentiment Analysis

The `pipeline()` function is the easiest way to use a Hugging Face model. You give it a **task name**, and it automatically:
1. Downloads a sensible pre-trained model for that task
2. Downloads the matching tokenizer
3. Wraps them so you can just pass in text and get an answer


In [ ]:
# ✍️ Code along: create your first pipeline!

from transformers import pipeline

classifier=pipeline('sentiment-analysis')

result=classifier("I absolutely loved this workshop!")
print(result)

**What just happened?**
- `pipeline("sentiment-analysis")` downloaded a pre-trained model (by default, a DistilBERT model fine-tuned on sentiment)
- It tokenized your sentence, ran it through the model, and converted the output into a human-readable label + confidence score


### 🙋 Try it yourself
Run the cell below and replace the text with your own sentences. Try something ambiguous or sarcastic — does it get it right?

In [ ]:
my_sentences = [
    "I can't believe how good this pizza is.",
    "This is the worst customer service I've ever experienced.",
    "I guess it was okay, nothing special.",
    # add your own sentence here!
]

results = classifier(my_sentences)
for sentence, result in zip(my_sentences, results):
    print(f"{sentence}\n  → {result['label']} (confidence: {result['score']:.2f})\n")


## ⭐ 4. A Tour of Pipelines

`pipeline()` supports dozens of tasks. Let's try a few of the most useful ones. For each, we'll run a demo, then you try it with your own input.

### 4.1 Text Generation
Give the model a prompt, and it predicts what comes next.

`max_length` -> Controls the maximum number of tokens in the generated sequence.
>💡 A larger max_length allows the model to generate a longer output.

`num_return_sequences`-> Controls how many different generated outputs we want.



In [ ]:
# ✍️ Code along

generator=pipeline("text-generation",model="gpt2")

output=generator("In the future, Aritificial intelligence will",max_length=40,num_return_sequences=1 )

output[0]["generated_text"]

🙋 **Try it:** Change the prompt below to something fun and re-run.

In [ ]:
# TODO: Add your own prompt and call generator() object to see the result



### 4.3 Zero-Shot Classification
This is a favorite — the model can classify text into categories **it was never explicitly trained on**, just by describing the categories in plain English.


In [ ]:
# ✍️ Code along

zero_shot = pipeline("zero-shot-classification")
text = "I need to fix a bug in my Python code before the deadline tonight."
candidate_labels = ["technology", "cooking", "sports", "politics"]

result = zero_shot(text, candidate_labels)
for label, score in zip(result["labels"], result["scores"]):
    print(f"{label}: {score:.2f}")



🙋 **Try it:** Change `text` and `candidate_labels` below to your own example.


In [ ]:
# TODO: Try your own text and candidate label

## ⭐ 5. Under the Hood, Part 1: Tokenizers

Models don't understand words — they understand **numbers**. A **tokenizer** converts text into numbers (and back).

In [ ]:
# ✍️ Code along

from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased-finetuned-sst-2-english")
text = "Transformers are amazing!"

tokens = tokenizer.tokenize(text)
print("Tokens:", tokens)

input_ids = tokenizer.convert_tokens_to_ids(tokens)
print("Token IDs:", input_ids)

encoded = tokenizer(text)
print("\nFull encoding:", encoded)


**Notice:**
- Words get split into smaller pieces (subwords) — this is how the model handles words it's never seen before
- `input_ids` are the numbers the model actually processes
- `attention_mask` tells the model which tokens are real vs. padding (useful when batching sentences of different lengths)

🙋 **Try it:** Tokenize your own sentence, including something long or unusual (like your name, or a made-up word).


In [ ]:
# TODO: Add your own sentence and check the output of tokenizer

## ⭐ 6. Under the Hood, Part 2: Models

Now let's load the actual model and run the full process manually — this is exactly what `pipeline()` was doing for us automatically.


The pipeline() function gives us a simple interface, but several steps are happening behind the scenes.

              pipeline("sentiment-analysis")
                          │
                          ↓
                 Load pretrained model
                          │
                          ↓
                    Load tokenizer
                          │
                          ↓
                    Input text
                          │
                          ↓
                      Tokenize
                          │
                          ↓
                   Token IDs + Mask
                          │
                          ↓
                  Transformer model
                          │
                          ↓
                       Logits
                          │
                          ↓
                 Prediction / Scores
                          │
                          ↓
                POSITIVE / NEGATIVE

In [ ]:
# ✍️ Code along

from transformers import AutoModelForSequenceClassification
import torch
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased-finetuned-sst-2-english"
)
text = "I really enjoyed learning about transformers today!"

inputs = tokenizer(text, return_tensors="pt")
print("Model input:", inputs)

with torch.no_grad():
    outputs = model(**inputs)

print("\nRaw output (logits):", outputs.logits)


Those raw numbers ("logits") aren't probabilities yet. We convert them using **softmax**:


In [ ]:
# ✍️ Code along

import torch.nn.functional as F

probabilities = F.softmax(outputs.logits, dim=-1)
print("Probabilities:", probabilities)

labels = model.config.id2label
for i, prob in enumerate(probabilities[0]):
    print(f"{labels[i]}: {prob.item():.2%}")


🎉 **You just reproduced what `pipeline("sentiment-analysis")` does automatically, step by step:**

1. Tokenizer: text → numbers
2. Model: numbers → raw logits
3. Softmax: logits → probabilities
4. Label mapping: probabilities → human-readable output


## ⭐ 7. Datasets Library

Models learn from data. Hugging Face hosts thousands of datasets you can load in one line. We'll use **Rotten Tomatoes**, a dataset of movie review snippets labeled positive (1) or negative (0).

In [ ]:
# ✍️ Code along

from datasets import load_dataset
dataset = load_dataset("cornell-movie-review-data/rotten_tomatoes")
print(dataset)


In [ ]:
# Look at a few examples
for i in range(3):
    example = dataset["train"][i]
    label = "positive" if example["label"] == 1 else "negative"
    print(f"[{label}] {example['text']}\n")


**What to notice:**
- The dataset is already split into `train`, `validation`, and `test`
- The **train** split is what the model learns from. The **test** split is held back to check how well it learned. It's like studying with practice problems vs. taking the real exam.
- Each example is a dictionary with a `text` and a `label`

🙋 **Try it:** How many examples are in the training split? (Hint: `len(dataset["train"])`)


## ⭐ 8. Fine-Tuning Your Own Model

So far we've used models exactly as someone else trained them.

**Fine-tuning** means taking a pre-trained model and training it a little more on *your* data so it gets better at *your* task.


> ⚠️ **Make sure your runtime is set to a T4 GPU** (Runtime → Change runtime type). Training on CPU will take much too long.

We'll fine-tune **DistilBERT** (a small, fast transformer) on a small slice of Rotten Tomatoes. The steps:
1. Pick a small subset of data (so training takes a couple of minutes)
2. Tokenize it
3. Load a fresh model
4. Measure accuracy **before** training
5. Train
6. Measure accuracy **after** training


### Step 1: Prepare a small subset


In [ ]:
train_data = dataset["train"].shuffle(seed=42).select(range(2000))
eval_data = dataset["test"].shuffle(seed=42).select(range(500))
print(len(train_data), "training examples,", len(eval_data), "evaluation examples")


### Step 2: Tokenize
We apply the tokenizer to every example. `batched=True` makes it fast.


In [ ]:
from transformers import AutoTokenizer

model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=128)

tokenized_train = train_data.map(tokenize, batched=True)
tokenized_eval = eval_data.map(tokenize, batched=True)
print(tokenized_train[0].keys())


### Step 3: Load a fresh model
We use `num_labels=2` because we have two classes (negative/positive). You'll see a warning that some weights are newly initialized. **That's expected!** The classification head on top is brand new and hasn't been trained yet.


In [ ]:
# ✍️ Code along

from transformers import AutoModelForSequenceClassification
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

### Step 4: Set up the Trainer
The `Trainer` class handles the training loop (batching, backpropagation, evaluation) so you don't have to write it yourself. We also define a simple **accuracy** metric.


In [ ]:
import numpy as np
from transformers import TrainingArguments, Trainer

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {"accuracy": (predictions == labels).mean()}

training_args = TrainingArguments(
    output_dir="my-finetuned-model",
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=25,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    compute_metrics=compute_metrics,
)


### Step 5: Accuracy BEFORE training
The model hasn't learned this task yet, so we expect roughly 50% (a coin flip).


In [ ]:
# ✍️ Code along

before = trainer.evaluate()
print(f"Accuracy before training: {before['eval_accuracy']:.2%}")


### Step 6: Train
This takes a few minutes on a T4 GPU. Watch the **loss** go down. That's the model learning.


In [ ]:
# ✍️ Code along — the big moment!

trainer.train()

### Step 7: Accuracy AFTER training


In [ ]:
# ✍️ Code along

after = trainer.evaluate()
print(f"Accuracy before: {before['eval_accuracy']:.2%}")
print(f"Accuracy after:  {after['eval_accuracy']:.2%}")

🎉 **You just fine-tuned a transformer.** With only 2,000 examples and a couple of minutes of training, accuracy should jump substantially. Let's try our new model on some sentences.


In [ ]:
from transformers import pipeline

my_pipeline = pipeline(
    "sentiment-analysis",
    model=trainer.model,
    tokenizer=tokenizer,
    device=trainer.model.device,
)

for text in ["A moving, beautifully acted film.", "Dull, predictable, and far too long."]:
    print(text, "->", my_pipeline(text))

> Note: the labels show up as `LABEL_0` / `LABEL_1`. That means 0 = negative and 1 = positive (as we saw in the dataset). You can set custom names via `id2label` when loading the model.


## ⭐ 10. Build a Web Demo with Gradio

A model in a notebook is great for you, but what if you want to show a friend or put it in your portfolio? **Gradio** turns a Python function into a web app in a few lines. This is also what powers Hugging Face **Spaces**.


In [ ]:
import gradio as gr

def analyze_sentiment(text):
    result = my_pipeline(text)[0]
    label = "Positive" if result["label"] == "LABEL_1" else "Negative"
    return {label: result["score"]}

demo = gr.Interface(
    fn=analyze_sentiment,
    inputs=gr.Textbox(lines=3, placeholder="Type a movie review..."),
    outputs=gr.Label(),
    title=" Movie Review Sentiment",
    description="A DistilBERT model fine-tuned by YOU at the ACM-W workshop!",
)

demo.launch(share=True)


`share=True` gives you a **public link** (valid for ~72 hours) that anyone can open, so send it to a friend!

### Making it permanent
To host a demo for free forever:
1. Create an account at [huggingface.co](https://huggingface.co)
2. Create a new **Space** and choose the Gradio SDK
3. Upload your `app.py` and a `requirements.txt`

🙋 **Try it:** Edit the `title` and `description`, add an example input using `examples=[...]`, and re-run.


## 12. 🎁 Bonus: Beyond Text *(~10 min)*

Transformers aren't just for text! The same `pipeline()` idea works for images, audio, and more.

### Image classification


In [ ]:
image_classifier = pipeline("image-classification")

url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/pipeline-cat-chonk.jpeg"
predictions = image_classifier(url)
for p in predictions[:3]:
    print(f"{p['label']}: {p['score']:.2f}")



### Zero-shot image classification with CLIP
Just like zero-shot text classification, but for images. You pick the labels in plain English.


In [ ]:
clip = pipeline("zero-shot-image-classification", model="openai/clip-vit-base-patch32")

result = clip(url, candidate_labels=["a cat", "a dog", "a car", "a plate of food"])
for r in result:
    print(f"{r['label']}: {r['score']:.2f}")


🙋 **Try it:** Find any image URL online (or upload a file to Colab) and try your own labels.

Other tasks to explore: `automatic-speech-recognition` (audio → text), `image-to-text` (captioning), `object-detection`. Browse them all at [huggingface.co/tasks](https://huggingface.co/tasks).


## 13. Wrap-up & Where to Go Next

### Next steps
- 📚 **[Hugging Face NLP Course](https://huggingface.co/learn/nlp-course)** — free, hands-on, goes from here through fine-tuning your own models
- 🎨 **[Hugging Face Spaces](https://huggingface.co/spaces)** — browse and build live ML demos, no server needed
- 🧪 **Go deeper on fine-tuning** — try a different dataset (e.g. `emotion`, `ag_news`), a bigger model, or more epochs, then `push_to_hub()` your result
- 🗂️ **[Datasets library](https://huggingface.co/docs/datasets)** — pairs with `transformers` for loading training data
- 💬 Join our ACM-W community to keep building!

---
### Thank you for coming! 💜
*Questions? Find us after the workshop or reach out via ACM-W.*